# Bài tập 1: Autoregressive Generation với RNN

Notebook này xây dựng mô hình **RNN sinh văn bản tự hồi quy** để tạo câu mới.

Chủ đề dữ liệu: **luyện tập thể thao**.

Các yêu cầu chính:

1. Dùng một đoạn văn bản ngắn khoảng 10-15 câu làm tập huấn luyện.
2. Sử dụng kiến trúc RNN đã học.
3. Lập trình hàm `generate(seed_text, max_len)`:
   - Đưa `seed_text` vào mô hình để dự đoán từ tiếp theo.
   - Lấy từ có xác suất cao nhất làm đầu vào tiếp theo.
   - Lặp lại cho đến khi đạt `max_len`.
4. So sánh ý nghĩa giữa **Teacher Forcing lúc học** và **Auto-regression lúc sinh**.

## Cell 1 - Import thư viện cần dùng

Ta sử dụng PyTorch để xây dựng và huấn luyện mô hình RNN.  
Tên biến và tên hàm được đặt tiếng Việt không dấu để dễ hiểu và dễ chạy.

In [ ]:
import re
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

thiet_bi = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị đang dùng:", thiet_bi)

Thiết bị đang dùng: cuda


## Cell 2 - Chuẩn bị dữ liệu văn bản ngắn

Dữ liệu gồm 12 câu ngắn về việc **luyện tập thể thao**.  
Văn bản được viết không dấu để việc tách từ và xây dựng từ điển đơn giản hơn.

In [ ]:
van_ban = """
moi buoi sang em chay bo quanh cong vien de ren suc ben
truoc khi tap luyen em khoi dong co the that ky
tap the thao deu dan giup co the khoe manh hon moi ngay
khi chay bo em giu nhip tho on dinh va buoc chan deu
sau gio hoc em danh cau long voi ban be trong san truong
boi loi giup em thu gian va tang suc khoe tim mach
tap gym can dung ky thuat de tranh chan thuong co bap
uong du nuoc trong luc tap giup co the khong bi met moi
ngu du giac giup co bap phuc hoi sau khi van dong
an uong lanh manh lam qua trinh luyen tap hieu qua hon
em dat muc tieu tap luyen nho de duy tri dong luc
the thao giup tinh than vui ve va hoc tap tap trung hon
"""

print(van_ban)


moi buoi sang em chay bo quanh cong vien de ren suc ben
truoc khi tap luyen em khoi dong co the that ky
tap the thao deu dan giup co the khoe manh hon moi ngay
khi chay bo em giu nhip tho on dinh va buoc chan deu
sau gio hoc em danh cau long voi ban be trong san truong
boi loi giup em thu gian va tang suc khoe tim mach
tap gym can dung ky thuat de tranh chan thuong co bap
uong du nuoc trong luc tap giup co the khong bi met moi
ngu du giac giup co bap phuc hoi sau khi van dong
an uong lanh manh lam qua trinh luyen tap hieu qua hon
em dat muc tieu tap luyen nho de duy tri dong luc
the thao giup tinh than vui ve va hoc tap tap trung hon



## Cell 3 - Tiền xử lý văn bản và xây dựng từ điển

Ở bước này, ta thực hiện:

- Đưa văn bản về chữ thường.
- Loại bỏ ký tự không cần thiết.
- Tách văn bản thành danh sách từ.
- Xây dựng từ điển `tu_sang_so`.
- Xây dựng từ điển ngược `so_sang_tu`.
- Chuyển toàn bộ văn bản thành chuỗi số.

In [ ]:
def tien_xu_ly_van_ban(van_ban):
    van_ban = van_ban.lower()
    van_ban = re.sub(r"[^a-zA-Z0-9\s]", " ", van_ban)
    danh_sach_tu = van_ban.split()
    return danh_sach_tu


danh_sach_tu = tien_xu_ly_van_ban(van_ban)

tu_khong_trung = sorted(set(danh_sach_tu))

tu_sang_so = {tu: chi_so for chi_so, tu in enumerate(tu_khong_trung)}
so_sang_tu = {chi_so: tu for tu, chi_so in tu_sang_so.items()}

chuoi_so = [tu_sang_so[tu] for tu in danh_sach_tu]
kich_thuoc_tu_dien = len(tu_sang_so)

print("Số lượng từ trong văn bản:", len(danh_sach_tu))
print("Kích thước từ điển:", kich_thuoc_tu_dien)

print("\nMột vài từ trong từ điển:")
print(list(tu_sang_so.items())[:20])

print("\nMột phần văn bản sau khi chuyển thành số:")
print(chuoi_so[:40])

Số lượng từ trong văn bản: 149
Kích thước từ điển: 94

Một vài từ trong từ điển:
[('an', 0), ('ban', 1), ('bap', 2), ('be', 3), ('ben', 4), ('bi', 5), ('bo', 6), ('boi', 7), ('buoc', 8), ('buoi', 9), ('can', 10), ('cau', 11), ('chan', 12), ('chay', 13), ('co', 14), ('cong', 15), ('dan', 16), ('danh', 17), ('dat', 18), ('de', 19)]

Một phần văn bản sau khi chuyển thành số:
[51, 9, 64, 26, 13, 6, 61, 15, 91, 19, 62, 66, 4, 85, 37, 68, 47, 26, 39, 22, 14, 72, 71, 41, 68, 72, 70, 20, 16, 31, 14, 72, 38, 49, 36, 51, 53, 37, 13, 6]


## Cell 4 - Tạo dữ liệu huấn luyện bằng Teacher Forcing

Trong lúc huấn luyện, mô hình học dự đoán từ tiếp theo bằng cách dùng **từ đúng hiện tại** trong dữ liệu gốc.

Ví dụ chuỗi từ:

```text
moi buoi sang em chay bo
```

Dữ liệu đầu vào:

```text
moi buoi sang em chay
```

Nhãn cần dự đoán:

```text
buoi sang em chay bo
```

Tức là:

- Nhận `moi` thì dự đoán `buoi`.
- Nhận `buoi` thì dự đoán `sang`.
- Nhận `sang` thì dự đoán `em`.

Cách tạo dữ liệu như vậy chính là **Teacher Forcing lúc học**.

In [ ]:
def tao_du_lieu_huan_luyen(chuoi_so, do_dai_ngu_canh):
    danh_sach_dau_vao = []
    danh_sach_nhan = []

    for vi_tri in range(len(chuoi_so) - do_dai_ngu_canh):
        dau_vao = chuoi_so[vi_tri : vi_tri + do_dai_ngu_canh]
        nhan = chuoi_so[vi_tri + 1 : vi_tri + do_dai_ngu_canh + 1]

        danh_sach_dau_vao.append(dau_vao)
        danh_sach_nhan.append(nhan)

    X = torch.tensor(danh_sach_dau_vao, dtype=torch.long)
    y = torch.tensor(danh_sach_nhan, dtype=torch.long)

    return X, y


do_dai_ngu_canh = 6

X, y = tao_du_lieu_huan_luyen(chuoi_so, do_dai_ngu_canh)

bo_du_lieu = TensorDataset(X, y)
bo_tai_du_lieu = DataLoader(bo_du_lieu, batch_size=4, shuffle=True)

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)

print("\nVí dụ X[0] dạng số:")
print(X[0])

print("\nVí dụ y[0] dạng số:")
print(y[0])

print("\nGiải mã X[0] thành từ:")
print([so_sang_tu[so.item()] for so in X[0]])

print("\nGiải mã y[0] thành từ:")
print([so_sang_tu[so.item()] for so in y[0]])

Kích thước X: torch.Size([143, 6])
Kích thước y: torch.Size([143, 6])

Ví dụ X[0] dạng số:
tensor([51,  9, 64, 26, 13,  6])

Ví dụ y[0] dạng số:
tensor([ 9, 64, 26, 13,  6, 61])

Giải mã X[0] thành từ:
['moi', 'buoi', 'sang', 'em', 'chay', 'bo']

Giải mã y[0] thành từ:
['buoi', 'sang', 'em', 'chay', 'bo', 'quanh']


## Cell 5 - Xây dựng mô hình RNN sinh văn bản

Mô hình gồm 3 phần chính:

1. `nn.Embedding`: chuyển chỉ số của từ thành vector nhúng.
2. `nn.RNN`: xử lý chuỗi từ theo từng bước thời gian.
3. `nn.Linear`: chuyển đầu ra RNN thành điểm dự đoán cho từng từ trong từ điển.

Đầu ra cuối cùng của mô hình có dạng:

```text
(batch_size, do_dai_ngu_canh, kich_thuoc_tu_dien)
```

Mỗi bước thời gian sẽ dự đoán từ tiếp theo tương ứng.

In [ ]:
class MoHinhRNNVanBan(nn.Module):
    def __init__(self, kich_thuoc_tu_dien, kich_thuoc_embedding, kich_thuoc_an):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=kich_thuoc_tu_dien,
            embedding_dim=kich_thuoc_embedding
        )

        self.rnn = nn.RNN(
            input_size=kich_thuoc_embedding,
            hidden_size=kich_thuoc_an,
            batch_first=True
        )

        self.fc = nn.Linear(
            in_features=kich_thuoc_an,
            out_features=kich_thuoc_tu_dien
        )

    def forward(self, dau_vao):
        vector_tu = self.embedding(dau_vao)
        dau_ra_rnn, trang_thai_an = self.rnn(vector_tu)
        du_doan = self.fc(dau_ra_rnn)

        return du_doan


kich_thuoc_embedding = 64
kich_thuoc_an = 64

mo_hinh = MoHinhRNNVanBan(
    kich_thuoc_tu_dien=kich_thuoc_tu_dien,
    kich_thuoc_embedding=kich_thuoc_embedding,
    kich_thuoc_an=kich_thuoc_an
).to(thiet_bi)

print(mo_hinh)

MoHinhRNNVanBan(
  (embedding): Embedding(94, 64)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=94, bias=True)
)


## Cell 6 - Huấn luyện mô hình

Ta dùng:

- `CrossEntropyLoss`: hàm mất mát cho bài toán dự đoán từ tiếp theo.
- `Adam`: thuật toán tối ưu trọng số.

Ở bước này, mô hình được huấn luyện bằng **Teacher Forcing** vì đầu vào là chuỗi từ đúng và nhãn là chuỗi từ đúng bị dịch sang phải 1 bước.

Khi gọi:

```python
loss.backward()
```

PyTorch sẽ thực hiện lan truyền ngược qua thời gian, tức **BPTT**, để cập nhật trọng số của RNN.

In [ ]:
ham_mat_mat = nn.CrossEntropyLoss()
bo_toi_uu = torch.optim.Adam(mo_hinh.parameters(), lr=0.01)

so_epoch = 60

for epoch in range(1, so_epoch + 1):
    mo_hinh.train()
    tong_loss = 0

    for dau_vao, nhan in bo_tai_du_lieu:
        dau_vao = dau_vao.to(thiet_bi)
        nhan = nhan.to(thiet_bi)

        du_doan = mo_hinh(dau_vao)

        loss = ham_mat_mat(
            du_doan.reshape(-1, kich_thuoc_tu_dien),
            nhan.reshape(-1)
        )

        bo_toi_uu.zero_grad()
        loss.backward()
        bo_toi_uu.step()

        tong_loss += loss.item()

    if epoch % 10 == 0:
        loss_trung_binh = tong_loss / len(bo_tai_du_lieu)
        print(f"Epoch {epoch:3d} | Loss trung bình: {loss_trung_binh:.4f}")

Epoch  10 | Loss trung bình: 0.2133
Epoch  20 | Loss trung bình: 0.1935
Epoch  30 | Loss trung bình: 0.1762
Epoch  40 | Loss trung bình: 0.1730
Epoch  50 | Loss trung bình: 0.2179
Epoch  60 | Loss trung bình: 0.2076


## Cell 7 - Hàm dự đoán từ tiếp theo

Hàm này nhận một đoạn văn bản ngắn, sau đó dự đoán **một từ tiếp theo**.

Ta lấy dự đoán ở bước thời gian cuối cùng, đưa qua `softmax`, rồi chọn từ có xác suất cao nhất.

In [ ]:
def du_doan_tu_tiep_theo(mo_hinh, seed_text):
    mo_hinh.eval()

    danh_sach_tu_seed = tien_xu_ly_van_ban(seed_text)

    danh_sach_so = []
    for tu in danh_sach_tu_seed:
        if tu in tu_sang_so:
            danh_sach_so.append(tu_sang_so[tu])

    if len(danh_sach_so) == 0:
        return None

    # Chỉ lấy các từ cuối cùng nếu seed_text dài hơn ngữ cảnh huấn luyện
    danh_sach_so = danh_sach_so[-do_dai_ngu_canh:]

    dau_vao = torch.tensor([danh_sach_so], dtype=torch.long).to(thiet_bi)

    with torch.no_grad():
        du_doan = mo_hinh(dau_vao)

        diem_tu_cuoi = du_doan[0, -1]
        xac_suat = torch.softmax(diem_tu_cuoi, dim=0)

        chi_so_tu_moi = torch.argmax(xac_suat).item()
        tu_moi = so_sang_tu[chi_so_tu_moi]

    return tu_moi


seed_thu = "em chay bo"
tu_tiep_theo = du_doan_tu_tiep_theo(mo_hinh, seed_thu)

print("Seed text:", seed_thu)
print("Từ tiếp theo dự đoán:", tu_tiep_theo)

Seed text: em chay bo
Từ tiếp theo dự đoán: quanh


## Cell 8 - Hàm generate(seed_text, max_len)

Đây là phần chính của bài **Auto-regressive Generation**.

Quy trình sinh văn bản:

1. Đưa `seed_text` vào mô hình.
2. Mô hình dự đoán từ tiếp theo.
3. Lấy từ có xác suất cao nhất.
4. Ghép từ đó vào cuối câu.
5. Dùng chính câu mới này làm đầu vào cho lần dự đoán tiếp theo.
6. Lặp lại cho đến khi sinh đủ `max_len` từ mới.

Điểm quan trọng:  
Lúc sinh văn bản, mô hình **không còn được đưa từ đúng từ dữ liệu gốc**, mà dùng lại từ do chính nó dự đoán. Đây gọi là **Auto-regression lúc sinh**.

In [ ]:
def generate(seed_text, max_len):
    cau_hien_tai = seed_text

    for _ in range(max_len):
        tu_moi = du_doan_tu_tiep_theo(mo_hinh, cau_hien_tai)

        if tu_moi is None:
            break

        cau_hien_tai = cau_hien_tai + " " + tu_moi

    return cau_hien_tai


seed_text = "tap the thao"
max_len = 12

van_ban_sinh = generate(seed_text, max_len)

print("Seed text:", seed_text)
print("Văn bản sinh ra:")
print(van_ban_sinh)

Seed text: tap the thao
Văn bản sinh ra:
tap the thao deu dan giup co the khoe manh hon moi ngay khi chay


## Cell 9 - Thử nghiệm với nhiều seed_text khác nhau

Ta thử sinh văn bản từ nhiều đoạn bắt đầu khác nhau để quan sát kết quả của mô hình.

In [ ]:
danh_sach_seed = [
    "moi buoi sang",
    "tap the thao",
    "sau gio hoc",
    "uong du nuoc",
    "the thao giup"
]

for seed in danh_sach_seed:
    ket_qua = generate(seed, max_len=10)
    print(f"Seed: {seed}")
    print(f"Kết quả: {ket_qua}")
    print("-" * 60)

Seed: moi buoi sang
Kết quả: moi buoi sang em chay bo quanh cong vien de ren suc ben
------------------------------------------------------------
Seed: tap the thao
Kết quả: tap the thao deu dan giup co the khoe manh hon moi ngay
------------------------------------------------------------
Seed: sau gio hoc
Kết quả: sau gio hoc em danh cau long voi ban be trong san truong
------------------------------------------------------------
Seed: uong du nuoc
Kết quả: uong du nuoc trong luc tap giup co the khong bi met moi
------------------------------------------------------------
Seed: the thao giup
Kết quả: the thao giup tinh than vui ve va hoc tap tap trung hon
------------------------------------------------------------


## Cell 10 - So sánh Teacher Forcing và Auto-regression

Trong bài này có 2 giai đoạn khác nhau:

### 1. Teacher Forcing lúc học

Khi huấn luyện, mô hình nhận chuỗi từ thật từ dữ liệu gốc.

Ví dụ:

```text
Đầu vào: moi buoi sang em chay
Nhãn:    buoi sang em chay bo
```

Mục tiêu là giúp mô hình học nhanh hơn và ổn định hơn vì mỗi bước đều có từ đúng làm đầu vào.

### 2. Auto-regression lúc sinh

Khi sinh văn bản bằng hàm `generate(seed_text, max_len)`, mô hình không có nhãn thật nữa.

Mô hình phải:

```text
dự đoán từ mới -> ghép vào câu -> dùng câu mới để dự đoán tiếp
```

Vì vậy, nếu mô hình dự đoán sai ở một bước, lỗi đó có thể ảnh hưởng đến các bước sinh sau.

In [ ]:
print("Teacher Forcing lúc học:")
print("Đầu vào mẫu:", [so_sang_tu[so.item()] for so in X[0]])
print("Nhãn mẫu:   ", [so_sang_tu[so.item()] for so in y[0]])

print("\nAuto-regression lúc sinh:")
seed_text = "em chay bo"
print("Seed ban đầu:", seed_text)

cau_hien_tai = seed_text
for buoc in range(1, 6):
    tu_moi = du_doan_tu_tiep_theo(mo_hinh, cau_hien_tai)
    cau_hien_tai = cau_hien_tai + " " + tu_moi
    print(f"Bước {buoc}: thêm từ '{tu_moi}' -> {cau_hien_tai}")

Teacher Forcing lúc học:
Đầu vào mẫu: ['moi', 'buoi', 'sang', 'em', 'chay', 'bo']
Nhãn mẫu:    ['buoi', 'sang', 'em', 'chay', 'bo', 'quanh']

Auto-regression lúc sinh:
Seed ban đầu: em chay bo
Bước 1: thêm từ 'quanh' -> em chay bo quanh
Bước 2: thêm từ 'cong' -> em chay bo quanh cong
Bước 3: thêm từ 'vien' -> em chay bo quanh cong vien
Bước 4: thêm từ 'de' -> em chay bo quanh cong vien de
Bước 5: thêm từ 'ren' -> em chay bo quanh cong vien de ren


## Cell 11 - Kết luận ngắn

Qua bài này, ta đã xây dựng được mô hình RNN sinh văn bản tự hồi quy.

Các yêu cầu chính đã hoàn thành:

- Có đoạn văn bản ngắn 12 câu về luyện tập thể thao.
- Có bước tiền xử lý, xây dựng từ điển và chuyển văn bản thành số.
- Có mô hình RNN dùng `nn.Embedding`, `nn.RNN`, `nn.Linear`.
- Có huấn luyện dự đoán từ tiếp theo bằng `CrossEntropyLoss`.
- Có Teacher Forcing trong lúc huấn luyện.
- Có hàm `generate(seed_text, max_len)` để sinh văn bản.
- Có quá trình sinh Auto-regression: từ dự đoán được dùng lại làm đầu vào cho bước sau.
- Có so sánh giữa Teacher Forcing lúc học và Auto-regression lúc sinh.